In [ ]:
import cv2
import mediapipe as mp
import time
import numpy as np

class SitToStandCounter:
    def __init__(self):
        # Initialize MediaPipe Pose
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        
        # Counter variables
        self.stand_count = 0
        self.current_state = "sitting"  # Start with sitting position
        self.test_duration = 30  # seconds
        self.start_time = None
        self.test_started = False
        self.test_stopped = False  # Flag for arm violation
        
        # Threshold for sitting vs standing
        self.sit_threshold = 0.35
        self.stand_threshold = 0.50
        
        # Arm position tracking
        self.arm_violation_count = 0
        self.arm_violation_threshold = 5  # Number of consecutive frames to confirm violation
        
        # CDC STEADI scoring norms (below average scores)
        self.scoring_norms = {
            'men': {
                '60-64': 14,
                '65-69': 12,
                '70-74': 12,
                '75-79': 11,
                '80-84': 10,
                '85-89': 8,
                '90-94': 7
            },
            'women': {
                '60-64': 12,
                '65-69': 11,
                '70-74': 10,
                '75-79': 10,
                '80-84': 9,
                '85-89': 8,
                '90-94': 4
            }
        }
        # Auto-start variables
        self.seated_time = None
        self.auto_start_enabled = True
        self.countdown_done = False
        self.countdown_start = None
        self.countdown_duration = 3  # seconds

    def is_user_seated(self, landmarks):
        """Detect if user is seated based on hip and knee positions."""
        try:
            left_hip = landmarks[self.mp_pose.PoseLandmark.LEFT_HIP.value].y
            right_hip = landmarks[self.mp_pose.PoseLandmark.RIGHT_HIP.value].y
            left_knee = landmarks[self.mp_pose.PoseLandmark.LEFT_KNEE.value].y
            right_knee = landmarks[self.mp_pose.PoseLandmark.RIGHT_KNEE.value].y

            # Average hip & knee positions
            hip_y = (left_hip + right_hip) / 2
            knee_y = (left_knee + right_knee) / 2

            # Sitting posture → knees LOWER than hips by a margin
            return knee_y > hip_y + 0.03

        except:
            return False
        
    def get_age_range(self, age):
        """Determine age range category"""
        if 60 <= age <= 64:
            return '60-64'
        elif 65 <= age <= 69:
            return '65-69'
        elif 70 <= age <= 74:
            return '70-74'
        elif 75 <= age <= 79:
            return '75-79'
        elif 80 <= age <= 84:
            return '80-84'
        elif 85 <= age <= 89:
            return '85-89'
        elif 90 <= age <= 94:
            return '90-94'
        else:
            return None
    
    def evaluate_score(self, count, age, gender):
        """
        Evaluate the score based on CDC STEADI norms
        Returns assessment and risk level
        """
        age_range = self.get_age_range(age)
        
        if age_range is None:
            return {
                'score': count,
                'age_range': 'Outside 60-94 range',
                'assessment': 'No norms available for this age',
                'risk_level': 'N/A'
            }
        
        gender_lower = gender.lower()
        if gender_lower not in ['men', 'women', 'male', 'female']:
            return {
                'score': count,
                'age_range': age_range,
                'assessment': 'Invalid gender specified',
                'risk_level': 'N/A'
            }
        
        # Normalize gender input
        if gender_lower in ['male', 'men']:
            gender_key = 'men'
        else:
            gender_key = 'women'
        
        threshold = self.scoring_norms[gender_key][age_range]
        
        if count < threshold:
            assessment = "BELOW AVERAGE - Indicates risk for falls"
            risk_level = "HIGH RISK"
        else:
            assessment = "AVERAGE or ABOVE - Good leg strength and endurance"
            risk_level = "LOW RISK"
        
        return {
            'score': count,
            'age_range': age_range,
            'threshold': threshold,
            'assessment': assessment,
            'risk_level': risk_level,
            'gender': gender_key.capitalize()
        }
    
    def print_final_report(self, count, age, gender, arm_violation=False):
        """Print formatted final report with scoring"""
        print("\n" + "="*70)
        print(" "*20 + "30-SECOND CHAIR STAND TEST")
        print(" "*25 + "FINAL RESULTS")
        print("="*70)
        
        result = self.evaluate_score(count, age, gender)
        
        print(f"\nPatient Information:")
        print(f"  Age: {age} years")
        print(f"  Gender: {result.get('gender', gender)}")
        print(f"  Age Range: {result['age_range']}")
        
        print(f"\nTest Results:")
        if arm_violation:
            print(f"  TEST STOPPED - Patient used arms to stand")
            print(f"  Total Stands: 0 (Protocol Violation)")
            print(f"\n  ⚠️  According to CDC STEADI protocol:")
            print(f"     'If the patient must use his/her arms to stand,")
            print(f"      stop the test. Record 0 for the number and score.'")
        else:
            print(f"  Total Stands in 30 seconds: {result['score']}")
            
            if 'threshold' in result:
                print(f"  Below Average Threshold: < {result['threshold']}")
            
            print(f"\nAssessment:")
            print(f"  {result['assessment']}")
            print(f"\nRisk Level: {result['risk_level']}")
        
        print("\n" + "-"*70)
        print("Reference: CDC STEADI - Stopping Elderly Accidents, Deaths & Injuries")
        print("Note: A below average score indicates a risk for falls.")
        print("="*70 + "\n")
        
        return result
    
    def check_arm_usage(self, landmarks):
        """
        Check if patient is using arms to help stand up
        Arms should be crossed on chest - check if hands move away from shoulders
        Returns: True if arm violation detected, False otherwise
        """
        try:
            # Get key landmarks
            left_shoulder = landmarks[self.mp_pose.PoseLandmark.LEFT_SHOULDER.value]
            right_shoulder = landmarks[self.mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
            left_wrist = landmarks[self.mp_pose.PoseLandmark.LEFT_WRIST.value]
            right_wrist = landmarks[self.mp_pose.PoseLandmark.RIGHT_WRIST.value]
            left_hip = landmarks[self.mp_pose.PoseLandmark.LEFT_HIP.value]
            right_hip = landmarks[self.mp_pose.PoseLandmark.RIGHT_HIP.value]
            
            # Calculate distances
            # In correct position, wrists should be near opposite shoulders (arms crossed)
            left_wrist_to_right_shoulder = np.sqrt(
                (left_wrist.x - right_shoulder.x)**2 + 
                (left_wrist.y - right_shoulder.y)**2
            )
            
            right_wrist_to_left_shoulder = np.sqrt(
                (right_wrist.x - left_shoulder.x)**2 + 
                (right_wrist.y - left_shoulder.y)**2
            )
            
            # Check if wrists are below hips (indicating pushing off chair or using armrests)
            left_wrist_below_hip = left_wrist.y > left_hip.y
            right_wrist_below_hip = right_wrist.y > right_hip.y
            
            # Calculate shoulder width for normalization
            shoulder_width = abs(left_shoulder.x - right_shoulder.x)
            
            # Normalized thresholds
            max_wrist_distance = shoulder_width * 1.8  # Wrists should stay relatively close to chest
            
            # Violation occurs if:
            # 1. Wrists are too far from shoulders (arms uncrossed)
            # 2. OR wrists are below hips (pushing off)
            arm_violation = (
                (left_wrist_to_right_shoulder > max_wrist_distance or 
                 right_wrist_to_left_shoulder > max_wrist_distance) or
                (left_wrist_below_hip or right_wrist_below_hip)
            )
            
            return arm_violation
            
        except Exception as e:
            print(f"Error checking arm usage: {e}")
            return False
        
    def calculate_body_posture(self, landmarks):
        """
        Calculate if person is sitting or standing based on body landmarks
        Returns: 'sitting', 'standing', or current state
        """
        try:
            # Get key landmarks
            left_shoulder = landmarks[self.mp_pose.PoseLandmark.LEFT_SHOULDER.value]
            right_shoulder = landmarks[self.mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
            left_hip = landmarks[self.mp_pose.PoseLandmark.LEFT_HIP.value]
            right_hip = landmarks[self.mp_pose.PoseLandmark.RIGHT_HIP.value]
            left_knee = landmarks[self.mp_pose.PoseLandmark.LEFT_KNEE.value]
            right_knee = landmarks[self.mp_pose.PoseLandmark.RIGHT_KNEE.value]
            
            # Calculate average positions
            shoulder_y = (left_shoulder.y + right_shoulder.y) / 2
            hip_y = (left_hip.y + right_hip.y) / 2
            knee_y = (left_knee.y + right_knee.y) / 2
            
            # Calculate hip-knee distance
            hip_knee_ratio = abs(hip_y - knee_y)
            
            # Calculate torso length
            torso_length = abs(shoulder_y - hip_y)
            
            # Normalized ratio (higher when standing, lower when sitting)
            if torso_length > 0:
                posture_ratio = hip_knee_ratio / torso_length
            else:
                return self.current_state
            
            # Determine state based on ratio
            if posture_ratio < self.sit_threshold:
                return "sitting"
            elif posture_ratio > self.stand_threshold:
                return "standing"
            else:
                return self.current_state
                
        except Exception as e:
            print(f"Error calculating posture: {e}")
            return self.current_state
    
    def update_count(self, new_state):
        """Update the stand count when transitioning from sitting to standing"""
        if self.current_state == "sitting" and new_state == "standing":
            self.stand_count += 1
            print(f"Stand detected! Count: {self.stand_count}")
        
        self.current_state = new_state
    
    def get_patient_info(self):
        """Get patient age and gender for scoring"""
        print("\n" + "="*70)
        print("Please enter patient information for scoring:")
        print("="*70)
        
        while True:
            try:
                age = int(input("Enter patient age (60-94): "))
                if age < 0:
                    print("Age must be positive. Please try again.")
                    continue
                break
            except ValueError:
                print("Invalid input. Please enter a number.")
        
        while True:
            gender = input("Enter patient gender (Male/Female or M/F): ").strip().lower()
            if gender in ['male', 'female', 'm', 'f', 'men', 'women']:
                if gender == 'm':
                    gender = 'male'
                elif gender == 'f':
                    gender = 'female'
                break
            else:
                print("Invalid input. Please enter Male/Female or M/F.")
        
        return age, gender
    
    def run_test(self):
        """Main function to run the 30-second chair stand test"""
        # 1. Ask for patient info first
        age, gender = 70, "F" #self.get_patient_info()

        # 2. Auto Select Camera 
        for i in range(5):  # Try camera indices 0 to 4
            cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
            if cap.isOpened():
                print(f"Camera found at index {i}")
                break
            cap.release()
        else:
            print("No camera found in indices 0-4")
            return
                
        print("=" * 70)
        print(" "*20 + "30-SECOND CHAIR STAND TEST")
        print("=" * 70)
        print("Instructions:")
        print("1. Sit in the middle of the chair")
        print("2. Cross your arms on your chest (hands on opposite shoulders)")
        print("3. Keep your feet flat on the floor")
        print("4. The test will start automatically once seated for 3 seconds")
        print("5. Stand up and sit down WITHOUT using your arms")
        print("6. ⚠️  If you use arms to stand, the test will stop (score = 0)")
        print("7. Press 'Q' to quit anytime")
        print("=" * 70)
        
        # Get patient information at the start
        age, gender = self.get_patient_info()
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            # Flip frame horizontally for mirror view
            frame = cv2.flip(frame, 1)
            
            # Convert to RGB for MediaPipe
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Process the frame
            results = self.pose.process(rgb_frame)
            
            # Draw pose landmarks
            if not results.pose_landmarks:
               cv2.imshow("30-Second Chair Stand Test", frame)
               if cv2.waitKey(1) & 0xFF == ord('q'):
                  break
               continue
        
            landmarks = results.pose_landmarks.landmark

            # ---------------- AUTO START LOGIC ----------------
            if self.auto_start_enabled and not self.test_started and not self.test_stopped:

               # If user is seated → start or continue timer
               if self.is_user_seated(landmarks):
                  if self.seated_time is None:
                     self.seated_time = time.time()
                     self.get_ready_start = time.time() #start 5 sec ready timer

                  # Display 5-second 'Get Ready' countdown 
                  ready_elapsed = time.time() - self.get_ready_start
                  ready_countdown = max(0, 5 - int(ready_elapsed))
                  cv2.putText(frame, "Get Ready!", (10, 120),
                              cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
                  cv2.putText(frame, f"Starting in: {ready_countdown}", (10, 160),
                              cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
                  cv2.putText(frame, "Sit in middle of chair, arms crossed, feet flat", (10, 200),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                  
                  
                  # After 5 seconds, start seated countdown for auto-start
                  if ready_elapsed >= 5:
                      seated_elapsed = time.time() - self.seated_time
                      cv2.putText(frame, f"Sit still: {max(0, 3 - int(seated_elapsed))}", (10, 200),
                                  cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)
                      cv2.putText(frame, "Maintain posture: arms crossed, feet flat", (10, 270),
                                  cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

                      if seated_elapsed >= 3 and not self.countdown_done:
                          self.countdown_start = time.time()
                          self.countdown_done = True
                          self.seated_time = None  # reset seated time
                          print("Seated countdown complete, starting test soon...")

               else:
                    # User moved before ready → reset timers
                    self.seated_time = None
                    self.countdown_done = False
                    self.countdown_start = None
                    self.get_ready_start = None

            # ---------------- COUNTDOWN DISPLAY ----------------
            if self.countdown_done and not self.test_started:
                elapsed = time.time() - self.countdown_start
                countdown = max(0, self.countdown_duration - int(elapsed))
                cv2.putText(frame, f"Starting in {countdown}", (10, 200),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
    
                if elapsed >= self.countdown_duration:
                    # Countdown finished → start test
                    self.start_time = time.time()
                    self.test_started = True
                    self.stand_count = 0
                    self.current_state = "sitting"
                    self.seated_time = None
                    self.countdown_done = False
                    print("TEST STARTED AUTOMATICALLY!")
            

            # ---------------- END AUTO START LOGIC ----------------
                
            # Check for arm usage and posture if test is running and not stopped
            if self.test_started and not self.test_stopped:
                    
                # Check for arm violation
                arm_violation = self.check_arm_usage(results.pose_landmarks.landmark)
                    
                if arm_violation:
                    self.arm_violation_count += 1
                else:
                    self.arm_violation_count = 0
                    
                # If violation detected for multiple consecutive frames, stop test
                if self.arm_violation_count >= self.arm_violation_threshold:
                    print("\n" + "="*70)
                    print("⚠️  TEST STOPPED - ARM USAGE DETECTED!")
                    print("Patient used arms to assist standing.")
                    print("According to CDC protocol, score = 0")
                    print("="*70)
                    self.test_stopped = True
                    self.stand_count = 0
                    self.test_started = False
                        
                    # Print final report with violation
                    self.print_final_report(0, age, gender, arm_violation=True)
                    print("Please sit again to start a new test automatically.")

                    # Calculate posture if test not stopped
                if self.test_started and not self.test_stopped:
                   new_state = self.calculate_body_posture(landmarks)
                   self.update_count(new_state)

            
            # Calculate remaining time
            if self.test_started and not self.test_stopped:
                elapsed_time = time.time() - self.start_time
                remaining_time = max(0, self.test_duration - elapsed_time)
                
                # Check if test is complete
                if remaining_time <= 0:
                    self.test_started = False
                    
                    # Print final report
                    self.print_final_report(self.stand_count, age, gender, arm_violation=False)
                    
            else:
                remaining_time = self.test_duration
            
            # Display information on frame
            if self.test_stopped:
                cv2.rectangle(frame, (5, 5), (500, 200), (0, 0, 128), -1)
                cv2.putText(frame, "TEST STOPPED", (10, 40),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
                cv2.putText(frame, "ARM USAGE DETECTED", (10, 85),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                cv2.putText(frame, "Score: 0", (10, 130),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                cv2.putText(frame, "Please sit again to retry", (10, 175),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            else:
                cv2.rectangle(frame, (5, 5), (400, 150), (0, 0, 0), -1)
                
                cv2.putText(frame, f"Count: {self.stand_count}", (10, 40),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
                cv2.putText(frame, f"Time: {remaining_time:.1f}s", (10, 85),
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)
                
                if not self.test_started:
                    cv2.putText(frame, "Please sit properly to start", (10, 130), 
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                else:
                    cv2.putText(frame, "TEST IN PROGRESS", (10, 130),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
            
            # Display frame
            cv2.imshow('30-Second Chair Stand Test', frame)
            
            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q') or key == ord('Q'):
                break
        # Cleanup
        cap.release()
        cv2.destroyAllWindows()
        self.pose.close()

# Run the test
if __name__ == "__main__":
    counter = SitToStandCounter()
    counter.run_test()

Camera found at index 0
                    30-SECOND CHAIR STAND TEST
Instructions:
1. Sit in the middle of the chair
2. Cross your arms on your chest (hands on opposite shoulders)
3. Keep your feet flat on the floor
4. The test will start automatically once seated for 3 seconds
5. Stand up and sit down WITHOUT using your arms
6. ⚠️  If you use arms to stand, the test will stop (score = 0)
7. Press 'Q' to quit anytime

Please enter patient information for scoring:
